Part A: the small-files problem

In [0]:
%python
for i in range(30):
    (spark.range(1000)
          .selectExpr(f"id + {i} * 1000 AS id", "round(rand() * 5000, 2) AS amount")
          .write.mode("append")
          .saveAsTable("workspace.finance.small_files_demo"))
print("done")

done


In [0]:
---Cell 2: count the files
DESCRIBE DETAIL workspace.finance.small_files_demo;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,c7f1bef1-0aae-44cd-a5ae-16d2b2865010,workspace.finance.small_files_demo,null,,2026-11-09T09:44:11.979Z,2026-11-09T09:45:13.000Z,List(),List(),17,136994,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, io.unitycatalog.tableId -> 51a0700e-374c-4e93-821f-2d40d40c02b1)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
OPTIMIZE workspace.finance.small_files_demo;

DESCRIBE DETAIL workspace.finance.small_files_demo;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,c7f1bef1-0aae-44cd-a5ae-16d2b2865010,workspace.finance.small_files_demo,null,,2026-11-09T09:44:11.979Z,2026-11-09T09:47:24.000Z,List(),List(),1,117902,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, io.unitycatalog.tableId -> 51a0700e-374c-4e93-821f-2d40d40c02b1)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


Part B: data skipping and liquid clustering

In [0]:
---Cell 4: generate 10 million transactions in two identical tables
CREATE OR REPLACE TABLE workspace.finance.perf_unclustered
TBLPROPERTIES ('delta.targetFileSize' = '16mb')
AS SELECT
  id                                                  AS txn_id,
  CAST(rand() * 1000 AS INT)                          AS user_id,
  date_add('2021-01-01', CAST(rand() * 1825 AS INT))  AS txn_date,
  element_at(array('Food & Dining','Groceries','Transport','Shopping',
                   'Entertainment','Utilities','Health','Travel'),
             CAST(rand() * 8 AS INT) + 1)             AS category,
  round(rand() * 5000, 2)                             AS amount
FROM range(10000000);

CREATE OR REPLACE TABLE workspace.finance.perf_clustered
CLUSTER BY (user_id)
TBLPROPERTIES ('delta.targetFileSize' = '16mb')
AS SELECT * FROM workspace.finance.perf_unclustered;

num_affected_rows,num_inserted_rows


In [0]:
---Cell 5: lay out both tables
OPTIMIZE workspace.finance.perf_unclustered;
OPTIMIZE workspace.finance.perf_clustered;

DESCRIBE DETAIL workspace.finance.perf_unclustered;
DESCRIBE DETAIL workspace.finance.perf_clustered;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,7442d7c2-f1a5-479e-a2ae-e05cc9fa763d,workspace.finance.perf_clustered,null,,2026-11-09T09:48:13.660Z,2026-11-09T09:49:02.000Z,List(),List(user_id),4,74915991,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, delta.enableRowTracking -> true, delta.checkpointPolicy -> v2, io.unitycatalog.tableId -> 13408cfc-4ea7-457f-8315-dcb6c4a860ef, delta.targetFileSize -> 16mb, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-4726c5fa-1d07-49ee-9846-df4b03948e4b, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-d4b52eaa-cf43-4e40-8f4f-1578b4f0acc6)",3,7,"List(appendOnly, clustering, deletionVectors, domainMetadata, invariants, rowTracking, v2Checkpoint)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
---Cell 6: the same query on both tables
SELECT category, count(*) AS txns, sum(amount) AS spent
FROM workspace.finance.perf_unclustered
WHERE user_id = 42
GROUP BY category;

SELECT category, count(*) AS txns, sum(amount) AS spent
FROM workspace.finance.perf_clustered
WHERE user_id = 42
GROUP BY category;

category,txns,spent
Food & Dining,1298,3196891.4600000023
Utilities,1277,3155662.549999995
Travel,1201,2936774.0800000024
Entertainment,1255,3182236.91
Shopping,1213,3049691.299999995
Groceries,1267,3139098.679999999
Transport,1198,2888218.5699999994
Health,1212,3062654.2500000023


Part D: VACUUM and its trade-off

In [0]:
---Cell 7: what’s in the history?

DESCRIBE HISTORY workspace.finance.small_files_demo;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
37,2026-11-09T09:47:24.000Z,76004538558006,mmaverick254@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> false, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(975018516772527),d87a77d0-5a12-420a-bfac-7353d3cf02de,1109-094359-ch2q34tf-v2n,36,SnapshotIsolation,false,"Map(numRemovedFiles -> 17, numRemovedBytes -> 136994, p25FileSize -> 117902, numDeletionVectorsRemoved -> 0, minFileSize -> 117902, numAddedFiles -> 1, maxFileSize -> 117902, p75FileSize -> 117902, p50FileSize -> 117902, numAddedBytes -> 117902)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
36,2026-11-09T09:45:13.001Z,76004538558006,mmaverick254@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(975018516772527),4960e126-05d4-4b24-8c19-a345af1d8390,1109-094359-ch2q34tf-v2n,34,WriteSerializable,true,"Map(numFiles -> 8, numOutputRows -> 1000, numOutputBytes -> 13445, conflictDetectionTimeMs -> 61)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
35,2026-11-09T09:45:13.000Z,76004538558006,mmaverick254@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(975018516772527),cfccc696-2298-4ccc-ba6a-0227f900c581,1109-094359-ch2q34tf-v2n,33,SnapshotIsolation,false,"Map(numRemovedFiles -> 33, numRemovedBytes -> 148265, p25FileSize -> 110113, numDeletionVectorsRemoved -> 0, conflictDetectionTimeMs -> 33, minFileSize -> 110113, numAddedFiles -> 1, maxFileSize -> 110113, p75FileSize -> 110113, p50FileSize -> 110113, numAddedBytes -> 110113)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
34,2026-11-09T09:45:11.000Z,76004538558006,mmaverick254@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(975018516772527),1907c485-e492-4089-8f72-a69a39f93ff7,1109-094359-ch2q34tf-v2n,33,WriteSerializable,true,"Map(numFiles -> 8, numOutputRows -> 1000, numOutputBytes -> 13436)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
33,2026-11-09T09:45:10.000Z,76004538558006,mmaverick254@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(975018516772527),cfccc696-2298-4ccc-ba6a-0227f900c581,1109-094359-ch2q34tf-v2n,32,WriteSerializable,true,"Map(numFiles -> 8, numOutputRows -> 1000, numOutputBytes -> 13417)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
32,2026-11-09T09:45:08.000Z,76004538558006,mmaverick254@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(975018516772527),e6483e0f-b272-433a-8a01-6a9ab0706dee,1109-094359-ch2q34tf-v2n,31,WriteSerializable,true,"Map(numFiles -> 8, numOutputRows -> 1000, numOutputBytes -> 13433)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
31,2026-11-09T09:45:07.000Z,76004538558006,mmaverick254@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(975018516772527),a777467a-f2f5-4155-a8b5-e9d43223f139,1109-094359-ch2q34tf-v2n,29,WriteSerializable,true,"Map(numFiles -> 8, numOutputRows -> 1000, numOutputBytes -> 13422, conflictDetectionTimeMs -> 35)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
30,2026-11-09T09:45:06.000Z,76004538558006,mmaverick254@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(975018516772527),ef6352db-1c10-47e8-b8d1-9723d8673621,1109-094359-ch2q34tf-v2n,28,SnapshotIsolation,false,"Map(numRemovedFiles -> 33, numRemovedBytes -> 132650, p25FileSize -> 94547, numDeletionVectorsRemoved -> 0, conflictDetectionTimeMs -> 65, minFileSize -> 94547, numAddedFiles -> 1, maxFileSize -> 94547, p75FileSize -> 94547, p50FileSize -> 94547, numAddedBytes -> 94547)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
29,2026-11-09T09:45:05.000Z,76004538558006,mmaverick254@gmail.com,WRITE,"

In [0]:
---Cell 8: what would VACUUM delete?
VACUUM workspace.finance.small_files_demo DRY RUN;

path


In [0]:
DESCRIBE TABLE EXTENDED workspace.finance.perf_clustered;

col_name,data_type,comment
txn_id,bigint,null
user_id,int,null
txn_date,date,null
category,string,null
amount,double,null
# Clustering Information,,
# col_name,data_type,comment
user_id,int,null
,,
# Delta Statistics Columns,,


In [0]:
DROP TABLE workspace.finance.small_files_demo;
DROP TABLE workspace.finance.perf_unclustered;
DROP TABLE workspace.finance.perf_clustered;